# ARIADNE Walkthrough

This notebook walks through the core pieces of the GPS-denied visual-inertial stack without requiring the full training run.

## 1. Load and visualize a satellite patch

If processed data is missing, we synthesize a small terrain tile inline.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path('..').resolve()
patches = list((ROOT / 'data' / 'processed' / 'satellite_patches').glob('patch_*.png'))

if patches:
    img = Image.open(patches[0])
    title = patches[0].name
else:
    rng = np.random.default_rng(0)
    arr = (rng.random((256, 256, 3)) * np.array([0.4, 0.55, 0.3]) * 255).astype(np.uint8)
    img = Image.fromarray(arr)
    title = 'synthetic_patch'

plt.figure(figsize=(4, 4))
plt.imshow(img)
plt.title(title)
plt.axis('off')
plt.show()

## 2. Generate a synthetic drone view

In [ ]:
from PIL import ImageEnhance, ImageFilter

drone = img.rotate(35, expand=False).resize((224, 224))
drone = ImageEnhance.Brightness(drone).enhance(0.85)
drone = drone.filter(ImageFilter.GaussianBlur(1.2))

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(img.resize((224, 224))); ax[0].set_title('satellite'); ax[0].axis('off')
ax[1].imshow(drone); ax[1].set_title('drone view'); ax[1].axis('off')
plt.show()

## 3. Embeddings and cosine similarity

In [ ]:
import torch
import torch.nn.functional as F
from torchvision import models, transforms

tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
backbone.fc = torch.nn.Identity()
backbone.eval()

with torch.no_grad():
    e1 = F.normalize(backbone(tf(img.convert('RGB')).unsqueeze(0)), dim=-1)
    e2 = F.normalize(backbone(tf(drone.convert('RGB')).unsqueeze(0)), dim=-1)
print('cosine similarity:', float((e1 @ e2.T).item()))

## 4. FAISS top-5 query (toy index)

In [ ]:
import faiss

xb = np.random.randn(200, 2048).astype('float32')
faiss.normalize_L2(xb)
index = faiss.IndexFlatIP(2048)
index.add(xb)
q = e1.numpy().astype('float32')
faiss.normalize_L2(q)
scores, idxs = index.search(q, 5)
print('top-5 indices', idxs[0])
print('scores', scores[0])
plt.bar(range(5), scores[0], color='#3b82f6')
plt.title('Toy FAISS similarities')
plt.show()

## 5. EKF filtered vs unfiltered path

In [ ]:
import sys
sys.path.insert(0, str(ROOT))
from model.ekf import VisualInertialEKF

truth, imu_only, fused = [], [], []
lat = lon = 48.4
ekf = VisualInertialEKF(lat, lon)
drift = VisualInertialEKF(lat, lon)
for i in range(200):
    lat += 0.00005
    lon += 0.00004
    imu = {'ax': 0.02, 'ay': 0.01, 'az': 9.81, 'gx': 0, 'gy': 0, 'gz': 0.002}
    drift.predict(imu, dt=0.1)
    ekf.predict(imu, dt=0.1)
    if i % 20 == 0:
        ekf.update(lat, lon, 0.85)
    truth.append((lat, lon))
    imu_only.append((drift.x[0], drift.x[1]))
    fused.append((ekf.x[0], ekf.x[1]))

truth = np.array(truth); imu_only = np.array(imu_only); fused = np.array(fused)
plt.figure(figsize=(6, 5))
plt.plot(truth[:,1], truth[:,0], color='#f59e0b', label='truth')
plt.plot(imu_only[:,1], imu_only[:,0], color='#ef4444', linestyle='--', label='IMU only')
plt.plot(fused[:,1], fused[:,0], color='#3b82f6', label='EKF fused')
plt.legend(); plt.xlabel('lon'); plt.ylabel('lat'); plt.title('EKF vs dead reckoning')
plt.show()

## 6. Error metrics sketch

After a full `make evaluate` run, inspect `model/checkpoints/error_histogram.png`. Below is a synthetic error distribution for illustration.

In [ ]:
errors = np.abs(np.random.normal(400, 250, size=500))
print('median m:', np.median(errors), 'mean m:', np.mean(errors))
plt.hist(errors, bins=40, color='#3b82f6')
plt.xlabel('error (m)'); plt.title('Illustrative localization errors')
plt.show()